# XLS-R + SVM

**Модель:** XLS-R-300M (frozen)  mean+std+max  SVM RBF

In [3]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import time
import torch
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import make_scorer, f1_score
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent.parent))

from shared import config, data_utils, train_utils
from shared.evaluate import find_optimal_threshold, evaluate
from shared.data_utils import build_feature_matrix, load_audio
from shared.results_utils import save_result_csv

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cpu


In [4]:
MODEL_ID = "facebook/wav2vec2-xls-r-300m"
xlsr_proc  = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
xlsr_model = Wav2Vec2Model.from_pretrained(MODEL_ID).to(DEVICE)
xlsr_model.eval()

def extract_xlsr(path):
    y, _ = load_audio(path, sr=16000)
    inp = xlsr_proc(y, sampling_rate=16000, return_tensors="pt", padding=True)
    with torch.no_grad():
        hs = xlsr_model(inp.input_values.to(DEVICE)).last_hidden_state[0].cpu().numpy()
    return np.concatenate([hs.mean(0), hs.std(0), hs.max(0)]).astype(np.float32)

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
(
    paths_trainval, labels_trainval, letters_trainval,
    paths_test, labels_test, letters_test,
) = data_utils.get_test_split()

print(f"Train+Val: {len(paths_trainval)} good: {np.sum(labels_trainval==0)}, bad: {np.sum(labels_trainval==1)}")
print(f"Test:      {len(paths_test)}  good: {np.sum(labels_test==0)},  bad: {np.sum(labels_test==1)}")

Train+Val: 2356 good: 1594, bad: 762
Test:      416  good: 281,  bad: 135


## Подбор гиперпараметров (GridSearchCV на train+val)

In [6]:
print("Извлечение признаков (train+val)...")
X_embeds_trainval = build_feature_matrix(paths_trainval, extract_xlsr, n_jobs=1)
print("Извлечение признаков (test)...")
X_embeds_test     = build_feature_matrix(paths_test,     extract_xlsr, n_jobs=1)

del xlsr_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

X_trainval = np.hstack([X_embeds_trainval, letters_trainval])
X_test     = np.hstack([X_embeds_test,     letters_test])


Извлечение признаков (train+val)...


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Извлечение признаков (test)...


In [7]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight="balanced",
                probability=True, random_state=config.RANDOM_STATE)),
])
f1_bad_scorer = make_scorer(f1_score, pos_label=config.CLASS_BAD, average="binary")
param_grid = {
    "clf__C":     [0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 50.0],
    "clf__gamma": ["scale", "auto", 0.001, 0.01, 0.05, 0.1],
}

t0 = time.perf_counter()
grid = GridSearchCV(pipeline, param_grid, cv=5,
                    scoring=f1_bad_scorer, refit=True, n_jobs=-1)
grid.fit(X_trainval, labels_trainval)
train_time_sec = time.perf_counter() - t0

best_params = grid.best_params_
best_clf = grid.best_estimator_
print(f"Лучшие параметры: {best_params}")

Лучшие параметры: {'clf__C': 1.0, 'clf__gamma': 'scale'}


## Оценка на test

In [8]:
y_proba = cross_val_predict(
    best_clf, X_trainval, labels_trainval, cv=5, method="predict_proba"
)[:, config.CLASS_BAD]
threshold = find_optimal_threshold(labels_trainval, y_proba)

y_proba_test = best_clf.predict_proba(X_test)[:, config.CLASS_BAD]
test_metrics = evaluate(labels_test, y_proba_test, threshold=threshold, verbose=True)
pd.DataFrame({
    "path":    paths_test,
    "y_true":  labels_test,
    "y_pred":  (y_proba_test >= threshold).astype(int),
    "y_proba": y_proba_test,
}).to_csv(exp_dir / "test_predictions.csv", index=False)

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_xlsr_svm",
    experiment_name="XLS-R (wav2vec2-xls-r-300m) + SVM",
    accuracy=test_metrics["accuracy"],
    f1_macro=test_metrics["f1_macro"],
    f1_bad=test_metrics["f1_bad"],
    roc_auc=test_metrics["roc_auc"],
    precision_bad=test_metrics["precision_bad"],
    recall_bad=test_metrics["recall_bad"],
    threshold=test_metrics["threshold"],
    embed_dim=3072,
    embed_dim_note="XLS-R-300M 1024-dim × 3 (mean+std+max)",
    notes=f"GridSearchCV cv=5 + threshold | best_params={best_params} | thr={threshold:.2f}",
    train_time_sec=train_time_sec,
)

              precision    recall  f1-score   support

        good       0.87      0.78      0.82       281
         bad       0.62      0.75      0.68       135

    accuracy                           0.77       416
   macro avg       0.74      0.77      0.75       416
weighted avg       0.79      0.77      0.78       416

Threshold : 0.30
Accuracy  : 0.7716
F1-macro  : 0.7513
F1-bad    : 0.6801
ROC-AUC   : 0.8265


PosixPath('/Users/dk/Desktop/ВШЭ/ВКР/HSE_VKR_DetectingSpeechDefects/experiments/03_pretrained/exp_xlsr_svm/result.csv')